# Student Performance Prediction: ML vs NN Comparison
## Using Actual Student Performance Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import sys
import os

# Add src to path
sys.path.append('../src')

from data_preprocessing import StudentDataPreprocessor
from model_training import StudentPerformanceModels

# Initialize
preprocessor = StudentDataPreprocessor()
model_trainer = StudentPerformanceModels()

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load and Explore Data

In [ ]:
# Load and preprocess actual data
print("Loading student performance data...")
df = preprocessor.load_and_preprocess_data('../data/raw/dst data.xlsx')

if df is not None:
    print(f"\nProcessed dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()[:10]}...")
    
    print("\nDataset Info:")
    print(df.info())
    
    print("\nFirst few rows:")
    display(df.head())
    
    print("\nStudent Performance Statistics:")
    print(f"Number of unique students: {df['StudentID'].nunique()}")
    print(f"Number of unique subjects: {df['SubjectCode'].nunique()}")
    print(f"Semesters available: {sorted(df['Semester'].unique())}")
    print(f"Pass rate: {df['pass_fail'].mean():.2%}")
else:
    print("Failed to load data. Please check the file path.")

## 2. Data Visualization

In [ ]:
if df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Semester-wise performance
    semester_performance = df.groupby('Semester')['TotalMarks'].mean()
    semester_performance.plot(kind='bar', ax=axes[0,0], color='skyblue')
    axes[0,0].set_title('Average Marks by Semester')
    axes[0,0].set_xlabel('Semester')
    axes[0,0].set_ylabel('Average Marks')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # Subject category performance
    subject_performance = df.groupby('subject_category')['TotalMarks'].mean().sort_values(ascending=False).head(10)
    subject_performance.plot(kind='bar', ax=axes[0,1], color='lightgreen')
    axes[0,1].set_title('Average Marks by Subject Category')
    axes[0,1].set_xlabel('Subject Category')
    axes[0,1].set_ylabel('Average Marks')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # Marks distribution
    axes[1,0].hist(df['TotalMarks'], bins=20, alpha=0.7, edgecolor='black', color='lightcoral')
    axes[1,0].axvline(df['TotalMarks'].mean(), color='red', linestyle='--', label=f'Mean: {df["TotalMarks"].mean():.1f}')
    axes[1,0].axvline(df['TotalMarks'].median(), color='blue', linestyle='--', label=f'Median: {df["TotalMarks"].median():.1f}')
    axes[1,0].set_title('Distribution of Total Marks')
    axes[1,0].set_xlabel('Total Marks')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].legend()
    
    # Pass/Fail distribution
    pass_fail_counts = df['pass_fail'].value_counts()
    axes[1,1].pie(pass_fail_counts, labels=['Fail', 'Pass'], autopct='%1.1f%%', 
                  colors=['lightcoral', 'lightgreen'], startangle=90)
    axes[1,1].set_title('Pass/Fail Distribution')
    
    plt.tight_layout()
    os.makedirs('../reports', exist_ok=True)
    plt.savefig('../reports/data_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Correlation heatmap
    plt.figure(figsize=(12, 10))
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    correlation_matrix = df[numeric_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Matrix of Numeric Features')
    plt.tight_layout()
    plt.savefig('../reports/correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

## 3. Prepare Data for Modeling

In [ ]:
if df is not None:
    print("\nPreparing data for modeling...")
    X, y, features = preprocessor.prepare_modeling_data(df, target_type='classification')
    
    if X is not None:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        print(f"\nTraining set: {X_train.shape}")
        print(f"Test set: {X_test.shape}")
        print(f"\nFeatures used: {features}")
        print(f"\nTarget distribution - Training:")
        print(f"Pass: {y_train.sum()} ({y_train.mean():.2%})")
        print(f"Fail: {len(y_train) - y_train.sum()} ({1 - y_train.mean():.2%})")
        print(f"\nTarget distribution - Test:")
        print(f"Pass: {y_test.sum()} ({y_test.mean():.2%})")
        print(f"Fail: {len(y_test) - y_test.sum()} ({1 - y_test.mean():.2%})")
    else:
        print("Failed to prepare modeling data")
        X_train = X_test = y_train = y_test = None

## 4. Train Random Forest Model

In [ ]:
if X_train is not None:
    print("\n" + "="*50)
    print("TRAINING RANDOM FOREST")
    print("="*50)
    rf_model, rf_importance, rf_pred = model_trainer.train_random_forest(X_train, X_test, y_train, y_test)
    
    if rf_importance is not None:
        print("\nTop 10 Feature Importance:")
        print(rf_importance.head(10))
        
        # Plot feature importance
        model_trainer.plot_feature_importance(rf_importance, top_n=10)
        
        # Save feature importance
        rf_importance.to_csv('../reports/feature_importance.csv', index=False)

## 5. Train Neural Network Model

In [ ]:
if X_train is not None:
    print("\n" + "="*50)
    print("TRAINING NEURAL NETWORK")
    print("="*50)
    nn_model, nn_history = model_trainer.train_neural_network(X_train, X_test, y_train, y_test)
    
    if nn_history is not None:
        # Plot training history
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(nn_history.history['accuracy'], label='Training Accuracy')
        plt.plot(nn_history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('Neural Network Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        plt.plot(nn_history.history['loss'], label='Training Loss')
        plt.plot(nn_history.history['val_loss'], label='Validation Loss')
        plt.title('Neural Network Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig('../reports/nn_training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

## 6. Compare Model Performance

In [ ]:
if X_train is not None and rf_model is not None and nn_model is not None:
    # Get predictions
    rf_pred = rf_model.predict(X_test)
    nn_pred = (nn_model.predict(X_test) > 0.5).astype(int).flatten()
    
    rf_accuracy = accuracy_score(y_test, rf_pred)
    nn_accuracy = accuracy_score(y_test, nn_pred)
    
    print("\n" + "="*50)
    print("MODEL COMPARISON")
    print("="*50)
    print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
    print(f"Neural Network Accuracy: {nn_accuracy:.4f}")
    
    # Create comparison visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Accuracy comparison
    models = ['Random Forest', 'Neural Network']
    accuracies = [rf_accuracy, nn_accuracy]
    axes[0].bar(models, accuracies, color=['#2E86AB', '#A23B72'])
    axes[0].set_ylim(0, 1)
    axes[0].set_title('Model Accuracy Comparison')
    axes[0].set_ylabel('Accuracy')
    for i, v in enumerate(accuracies):
        axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')
    
    # Confusion Matrix - Random Forest
    cm_rf = confusion_matrix(y_test, rf_pred)
    sns.heatmap(cm_rf, annot=True, fmt='d', ax=axes[1], cmap='Blues')
    axes[1].set_title('Random Forest Confusion Matrix')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    
    # Confusion Matrix - Neural Network
    cm_nn = confusion_matrix(y_test, nn_pred)
    sns.heatmap(cm_nn, annot=True, fmt='d', ax=axes[2], cmap='Blues')
    axes[2].set_title('Neural Network Confusion Matrix')
    axes[2].set_xlabel('Predicted')
    axes[2].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.savefig('../reports/model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. Save Models

In [ ]:
if rf_model is not None:
    os.makedirs('../models', exist_ok=True)
    model_trainer.save_model(rf_model, 'random_forest', '../models/random_forest.pkl')
    
if nn_model is not None:
    model_trainer.save_model(nn_model, 'neural_network', '../models/grade_nn.h5')
    
print("\nModels saved successfully!")
print("Random Forest: models/random_forest.pkl")
print("Neural Network: models/grade_nn.h5")

## 8. Train LSTM for Sequential Prediction

In [ ]:
if df is not None:
    print("\n" + "="*50)
    print("TRAINING LSTM FOR SEQUENTIAL PREDICTION")
    print("="*50)
    
    X_seq, y_seq = preprocessor.prepare_sequential_data(df, sequence_length=2)
    
    if len(X_seq) > 0:
        print(f"Sequential data shape: X_seq={X_seq.shape}, y_seq={y_seq.shape}")
        
        X_seq_train, X_seq_test, y_seq_train, y_seq_test = train_test_split(
            X_seq, y_seq, test_size=0.2, random_state=42
        )
        
        lstm_model, lstm_history = model_trainer.train_lstm(X_seq_train, X_seq_test, y_seq_train, y_seq_test)
        
        if lstm_model is not None:
            model_trainer.save_model(lstm_model, 'lstm', '../models/lstm_model.h5')
            print("LSTM model saved: models/lstm_model.h5")
            
            # Plot LSTM training history
            if lstm_history is not None:
                plt.figure(figsize=(12, 4))
                
                plt.subplot(1, 2, 1)
                plt.plot(lstm_history.history['loss'], label='Training Loss')
                plt.plot(lstm_history.history['val_loss'], label='Validation Loss')
                plt.title('LSTM Training Loss')
                plt.xlabel('Epoch')
                plt.ylabel('Loss')
                plt.legend()
                
                plt.subplot(1, 2, 2)
                plt.plot(lstm_history.history['mae'], label='Training MAE')
                plt.plot(lstm_history.history['val_mae'], label='Validation MAE')
                plt.title('LSTM Training MAE')
                plt.xlabel('Epoch')
                plt.ylabel('MAE')
                plt.legend()
                
                plt.tight_layout()
                plt.savefig('../reports/lstm_training_history.png', dpi=300, bbox_inches='tight')
                plt.show()
    else:
        print("Not enough sequential data for LSTM training")

print("\nProject completed successfully!")